# Module 3: Inference Optimization + Model Export

A model that trains well is only half the story. In production, **inference** is what users interact with.

In this module, you'll:

1. **Benchmark** baseline inference latency
2. **Quantize** the model (dynamic quantization) for smaller size and faster CPU inference
3. **Export** to TorchScript and ONNX for deployment outside Python
4. **Compare** all approaches on latency and model size

---

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import torch
import torch.nn as nn
from pathlib import Path

from src.model import build_model
from src.data import prepare_wikitext2
from src.evaluate import benchmark_inference, generate_sample
from src.export import export_torchscript, export_onnx, verify_torchscript, verify_onnx
from src.utils import set_seed, get_device, CheckpointManager

set_seed(42)
device = get_device()
print(f"Device: {device}")

In [ ]:
# Load trained model from checkpoint
_, _, _, tokenizer = prepare_wikitext2(
    vocab_size=8192, seq_len=128, tokenizer_path='../tokenizer.json'
)

config = {
    'vocab_size': tokenizer.get_vocab_size(),
    'd_model': 256, 'n_heads': 4, 'd_ff': 512,
    'n_layers': 4, 'max_seq_len': 128, 'dropout': 0.1,
}

model = build_model(config).to(device)

# Try loading checkpoint from Module 1; if not available, use untrained
ckpt_manager = CheckpointManager('../checkpoints')
latest = ckpt_manager.latest()
if latest:
    ckpt_manager.load(model, path=latest)
    print(f"Loaded checkpoint: {latest}")
else:
    print("No checkpoint found — using untrained model (results still demonstrate the patterns)")

model.eval()
print(f"Model: {model.count_parameters():,} parameters")

## 3.1 Baseline Inference Benchmark

Before optimizing, measure where you start. We'll benchmark:
- Single sample latency
- Batched inference throughput
- Text generation latency

In [ ]:
# Create test inputs
seq_len = 128
single_input = torch.randint(0, config['vocab_size'], (1, seq_len)).to(device)
batch_input = torch.randint(0, config['vocab_size'], (32, seq_len)).to(device)

# Benchmark single inference
single_stats = benchmark_inference(model, single_input, device, n_runs=200)
print("=== Single Sample Inference ===")
print(f"  Mean: {single_stats['mean_ms']:.2f} ms")
print(f"  P50:  {single_stats['p50_ms']:.2f} ms")
print(f"  P95:  {single_stats['p95_ms']:.2f} ms")
print(f"  P99:  {single_stats['p99_ms']:.2f} ms")

# Benchmark batched inference
batch_stats = benchmark_inference(model, batch_input, device, n_runs=100)
print(f"\n=== Batched Inference (batch_size=32) ===")
print(f"  Mean: {batch_stats['mean_ms']:.2f} ms")
print(f"  Per sample: {batch_stats['mean_ms'] / 32:.2f} ms")
print(f"  Throughput: {32 / batch_stats['mean_ms'] * 1000:.0f} samples/sec")

## 3.2 Dynamic Quantization

Quantization reduces model precision from float32 to int8, cutting model size by ~4x and improving CPU inference speed.

**Dynamic quantization** is the easiest form — it quantizes weights ahead of time and activations on-the-fly.

Best for: **CPU inference** and **linear-heavy models** (like transformers).

In [ ]:
# Move to CPU for quantization
model_cpu = build_model(config)
if latest:
    ckpt_manager.load(model_cpu, path=latest)
model_cpu.eval()

# Dynamic quantization
model_quantized = torch.ao.quantization.quantize_dynamic(
    model_cpu,
    {nn.Linear},  # Quantize Linear layers
    dtype=torch.qint8,
)

# Compare sizes
import tempfile, os

def model_size_mb(model):
    with tempfile.NamedTemporaryFile(delete=False) as f:
        torch.save(model.state_dict(), f.name)
        size = os.path.getsize(f.name) / 1024 / 1024
        os.unlink(f.name)
    return size

orig_size = model_size_mb(model_cpu)
quant_size = model_size_mb(model_quantized)

print(f"Original model:   {orig_size:.1f} MB")
print(f"Quantized model:  {quant_size:.1f} MB")
print(f"Size reduction:   {(1 - quant_size/orig_size) * 100:.0f}%")

In [ ]:
# Benchmark quantized vs original on CPU
cpu_device = torch.device('cpu')
test_input_cpu = torch.randint(0, config['vocab_size'], (1, seq_len))

orig_stats = benchmark_inference(model_cpu, test_input_cpu, cpu_device, n_runs=200)
quant_stats = benchmark_inference(model_quantized, test_input_cpu, cpu_device, n_runs=200)

print(f"{'Metric':<15} {'Original':>12} {'Quantized':>12} {'Speedup':>10}")
print('-' * 52)
for metric in ['mean_ms', 'p50_ms', 'p95_ms']:
    orig_val = orig_stats[metric]
    quant_val = quant_stats[metric]
    speedup = orig_val / quant_val
    label = metric.replace('_ms', '')
    print(f"{label:<15} {orig_val:>10.2f}ms {quant_val:>10.2f}ms {speedup:>9.2f}x")

## 3.3 TorchScript Export

TorchScript converts your model to an intermediate representation that can be:
- Loaded in C++ (no Python dependency)
- Serialized and shipped as a single file
- Optimized by the TorchScript JIT compiler

Two approaches:
- **`torch.jit.trace`**: Records operations during a forward pass (simpler, but can't handle dynamic control flow)
- **`torch.jit.script`**: Compiles Python code to TorchScript IR (handles control flow, but more restrictions)

In [ ]:
# We'll use a wrapper that only returns logits (TorchScript needs simple return types)
class InferenceWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        return self.model(input_ids)['logits']

wrapper = InferenceWrapper(model_cpu)
wrapper.eval()

sample_input = torch.randint(0, config['vocab_size'], (1, seq_len))

# Export via tracing
traced_path = export_torchscript(
    wrapper, sample_input,
    output_path='../exports/model_traced.pt',
    method='trace',
)

# Verify
verify_torchscript(wrapper, traced_path, sample_input)

In [ ]:
# Load and benchmark TorchScript model
ts_model = torch.jit.load(traced_path)
ts_model.eval()

ts_stats = benchmark_inference(ts_model, test_input_cpu, cpu_device, n_runs=200)

print(f"TorchScript mean latency: {ts_stats['mean_ms']:.2f} ms")
print(f"Original mean latency:    {orig_stats['mean_ms']:.2f} ms")
print(f"Speedup: {orig_stats['mean_ms'] / ts_stats['mean_ms']:.2f}x")

## 3.4 ONNX Export

ONNX (Open Neural Network Exchange) is an open format for ML models. It lets you run inference using:
- **ONNX Runtime** (optimized for CPU/GPU)
- **TensorRT** (NVIDIA GPU optimization)
- **OpenVINO** (Intel hardware)
- **CoreML** (Apple devices)

ONNX Runtime often provides the best CPU inference performance.

In [ ]:
# Export to ONNX
onnx_path = export_onnx(
    wrapper, sample_input,
    output_path='../exports/model.onnx',
)

# Verify
try:
    verify_onnx(wrapper, onnx_path, sample_input)
except ImportError:
    print("onnxruntime not installed — install with: pip install onnxruntime")

In [ ]:
# Benchmark ONNX Runtime inference
try:
    import onnxruntime as ort
    import numpy as np
    
    session = ort.InferenceSession(onnx_path)
    
    # Warmup
    np_input = test_input_cpu.numpy()
    for _ in range(10):
        session.run(None, {'input_ids': np_input})
    
    # Benchmark
    latencies = []
    for _ in range(200):
        start = time.perf_counter()
        session.run(None, {'input_ids': np_input})
        latencies.append((time.perf_counter() - start) * 1000)
    
    latencies.sort()
    print(f"ONNX Runtime mean latency: {sum(latencies)/len(latencies):.2f} ms")
    print(f"ONNX Runtime P50:          {latencies[len(latencies)//2]:.2f} ms")
    print(f"PyTorch mean latency:      {orig_stats['mean_ms']:.2f} ms")
    print(f"Speedup: {orig_stats['mean_ms'] / (sum(latencies)/len(latencies)):.2f}x")
    
except ImportError:
    print("onnxruntime not installed — skipping benchmark")

## 3.5 Comparison Summary

Let's put all approaches side by side.

In [ ]:
# Collect all results
results = {
    'PyTorch (FP32)': {
        'latency_ms': orig_stats['mean_ms'],
        'size_mb': orig_size,
    },
    'Quantized (INT8)': {
        'latency_ms': quant_stats['mean_ms'],
        'size_mb': quant_size,
    },
    'TorchScript': {
        'latency_ms': ts_stats['mean_ms'],
        'size_mb': Path(traced_path).stat().st_size / 1024 / 1024,
    },
}

try:
    results['ONNX Runtime'] = {
        'latency_ms': sum(latencies) / len(latencies),
        'size_mb': Path(onnx_path).stat().st_size / 1024 / 1024,
    }
except NameError:
    pass

baseline_latency = results['PyTorch (FP32)']['latency_ms']

print(f"{'Format':<20} {'Latency (ms)':>14} {'Size (MB)':>10} {'Speedup':>10}")
print('=' * 58)
for name, data in results.items():
    speedup = baseline_latency / data['latency_ms']
    print(f"{name:<20} {data['latency_ms']:>12.2f}ms {data['size_mb']:>8.1f}MB {speedup:>9.2f}x")

## Key Takeaways

| Export format | Best for | Trade-off |
|--------------|---------|----------|
| **PyTorch eager** | Prototyping, debugging | Slowest, most flexible |
| **Dynamic quantization** | CPU deployment | 2-4x smaller, 1.5-3x faster on CPU |
| **TorchScript** | C++ deployment, mobile | No Python needed |
| **ONNX** | Multi-platform inference | Best CPU perf with ORT |

**Production recommendation**: Use ONNX Runtime for CPU serving, quantized models for edge/mobile, and keep PyTorch eager for GPU serving (it's already fast on CUDA).

**Next up**: Module 4 — deploying the model as a FastAPI service with Docker and Cloud Run.